# O Problema de Localização de Facilidades




In [1]:
#ao executar este código, instalamos o suporte ao AMPL no notebook e no python
!pip install -q amplpy
from amplpy import tools
ampl = tools.ampl_notebook(
    modules=["highs", "coin"], # pick from over 20 modules including most commercial and open-source solvers
    license_uuid="bcc3d88a-8b8c-4c93-8f21-c2bedd3fc48f") # your license UUID

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 13.6 MB/s eta 0:00:00
Licensed to AMPL Community Edition License for <santi.everton@gmail.com>.


## Caso trabalhássemos para o INEP, como poderíamos auxiliar no processo de escolha dos locais de prova do ENEM?

### O que sabemos sobre o problema:

1. Existe um conjunto de candidatos fazendo a prova;
2. Existe um conjunto de locais a disposição do INEP onde a prova pode ser aplicada;
3. Cada local de prova tem uma capacidade;
4. O número de equipes que o INEP dispõe para aplicar a prova limita o número de locais de podem ser usados;
5. Sabemos a que distância está cada possível local de prova emrelação a cada candidato;

> **Simplificação** Custos fixos e variáveis poderiam ser usados também para guiar a tomada de decisão, como pagamento diárias às pessoas que aplicarão à prova e aluguel de imóveis. Também poderíamos levar em conta o número de salas de aula por imóveis, número de pessoas necessárias para cuidar a prova em cada sala, fiscais de corredor, dentre outros recursos relacionados à logística de aplicação da prova. Porém, vamos desconsiderar todos estes pontos para facilitar a modelagem e entendimento do modelo a ser criado.

### O que queremos saber (decidir):

1. Dado que não podemos utilizar todos os locais disponíveis para aplicar a prova, precisamos decidir quais serão utilizados;
2. Dado que cada local de prova tem uma capacidade, precisamos decidir quantos e quais condadidatos farão a prova em cada um dos lugares escolhidos;

### Nosso objetivo poderia ser:

* Minimizar a distância percorrida pelo candidato que mais precisará se deslocar;
* Minimizar o somatório de todas as distâncias percorridas por todos os candidatos;

### Dados históricos

Número de inscritos na prova em edições passadas.

|Ano|Inscritos|
|---|---|
|2022 |	3.396.597|
|2021 |	3.109.800|
|2020 |	5.616.115|
|2019 |	5.095.338|
|2018 |	5.554.790|
|2017 |	6.763.122|
|2016 |	8.681.686|
|2015 |	7.792.024|
|2014 |	8.722.283|
|2013 |	7.153.577|
|2012 |	5.814.644|
|2011 |	5.380.857|
|2010 |	4.626.096|
|2009| 	4.148.721|

## Modelando o problema

Listar os parâmetros e variáveis de decisão, para depois produzir a formulação. Não avance direto para a codificação.

## Dados de teste baixados do GoogleDrive

O código a seguir baixa os arquivos contendo os dados do problema que estão zipados e salvos no GoogleDrive, extraindo-os para a pasta 'dados'.

Após a extração dos arquivos, note que temos 4 casos de teste.

Note ainda que são arquivos no formato .txt, não .dat. O formato de cada arquivo segue o padrão:

```
<n_candidatos> <n_locais> <n_equipes_aplicacao>
<capacidade_local_1 ... capacidade_local_n>
<distancia_candidato_1-local-1 ... distancia_candidato_1-local-n>
...
<distancia_candidato_m-local-1 ... distancia_candidato_m-local-n>

```


In [3]:
!rm -r opt_05_facilidades.zip
!gdown https://drive.google.com/uc?id=1vDOmeVKcJh--Xlr32PdTeIXLqg09uMav
!unzip -qq opt_05_facilidades.zip -d ./dados/

rm: cannot remove 'opt_05_facilidades.zip': No such file or directory
Downloading...
From: https://drive.google.com/uc?id=1vDOmeVKcJh--Xlr32PdTeIXLqg09uMav
To: /content/opt_05_facilidades.zip
100% 670k/670k [00:00<00:00, 128MB/s]


## Vamos usar Python e AMPL nesta empreitada

Passos:

1. Formular o problema como um problema de **Programação Linear Inteira**;

2. Transcrever para **AMPL**;

3. Criar uma **função** que recebe como argumento o nome do arquivo de dados a ser aberto e retorna um dicionário com os dados deste arquivo prontos para consumo. Poderia pensar em retornar vetores e matrizes Numpy, ou mesmo ler um DataFrame se fosse o caso.

4. Escreva um **programa em Python** que, usando o módulo do AMPL, carrega o modelo e mostra sua solução;

5. Ao final reflita, o que poderíamos fazer com os recursos que aprendemos a utilizar hoje?


## Passo 1: Formulação




## Passo 2: Transcrever modelo para AMPL

In [10]:
%%writefile facility.mod
param m > 0;
param n > 0;
param p > 0;
param d{1..m,1..n} >= 0;
param c{1..n} >=0;
var x{1..m,1..n} binary;
var y{1..n} binary;
var z >=0;
minimize custo: sum{i in 1..m, j in 1..n}d[i,j]*x[i,j];
subject to r1: sum{j in 1..n}y[j]==p;
subject to r2{j in 1..n}: sum{i in 1..m}x[i,j] <= c[j];
subject to r3{i in 1..m, j in 1..n}:x[i,j] <= y[j];
subject to r4{i in 1..m, j in 1..n}:z>=d[i,j]*x[i,j];


Overwriting facility.mod


## Passo 3: Criar função em Python para ler os dados e retorná-los em um dicionário

In [6]:
def ler_dados(caminho_arquivo):
    with open(caminho_arquivo, "r") as f:
        primeira = list(map(int, f.readline().split(" ")))
        m, n, p = primeira
        c = list(map(int, f.readline().replace("\t", " ").split(" ")))
        d = []
        for i in range(m):
            linha = list(map(float, f.readline().replace("\t", " ").split(" ")))
            d.append(linha)
    return {
        'm': m,
        'n': n,
        'p': p,
        'c': c,
        'd': d,
    }
dados = ler_dados("dados/ilustrativo1.txt")
print(dados)

{'m': 10, 'n': 5, 'p': 3, 'c': [2, 3, 5, 1, 4], 'd': [[62.7, 111.19, 59.65, 97.79, 4.73], [9.23, 43.95, 39.23, 108.7, 29.87], [75.03, 16.65, 82.34, 69.12, 106.08], [10.55, 49.68, 89.62, 17.12, 32.66], [71.14, 1.71, 20.23, 94.24, 45.63], [64.42, 16.07, 32.04, 86.13, 17.61], [16.26, 22.26, 21.66, 80.02, 43.82], [96.15, 50.87, 94.72, 23.89, 24.13], [5.29, 53.94, 93.37, 41.65, 83.57], [25.02, 53.57, 32.37, 71.82, 52.1]]}


## Passo 4: Escrever um programa em Python que resolve nosso modelo usando dados provenientes de um arquivo

> Dica: se usarmos um DataFrame para ler e exportar uma matriz para o AMPL, podemos usar o método unstack para ganho de performance. O código que mostrarei abaixo é um tanto lento no trato de muitos dados.

In [23]:
"""
(colinha)

métodos do ampl:

    AMPL(): instanciar o AMPL em uma variável

    .set_option(<option_str>, <value_str>): definir solver

    .eval(<command_str>): avaliar um comando ampl. Ex. carregar modelo.

    .param[<param_key>] = <param_value>: setar o valor de um parâmetro

    .solve(): resolver modelo

    .solve(problem='', solver='', verbose=True, return_output=False,

    .get_objective(<name_str>): obter função objetivo e .value() seu valor

    .get_variable(<var_name_str>).get_values().to_list(): obter lista com os
        valores das variáveis

    obs1: matrizes e vetores são informados como dicionários

"""
from amplpy import AMPL
import pandas as pd
dados = ler_dados("dados/ilustrativo3.txt")
ampl = AMPL()
#ampl.eval("model facility.mod");
ampl.eval("""
            problem facility;
            param m > 0;
            param n > 0;
            param p > 0;
            param d{1..m,1..n} >= 0;
            param c{1..n} >=0;
            var x{1..m,1..n} binary;
            var y{1..n} binary;
            var z >=0;
            minimize custo: z;
            subject to r1: sum{j in 1..n}y[j]==p;
            subject to r2{j in 1..n}: sum{i in 1..m}x[i,j] <= c[j];
            subject to r3{i in 1..m, j in 1..n}:x[i,j] <= y[j];
            subject to r4{i in 1..m, j in 1..n}:z>=d[i,j]*x[i,j];
            subject to r5{i in 1..m}:sum{j in 1..n}x[i,j] == 1;
            """)
ampl.param["m"] = dados["m"]
ampl.param["n"] = dados["n"]
ampl.param["p"] = dados["p"]
ampl.param["c"] = {(j+1):valor for j,valor in enumerate(dados["c"])}
ampl.param["d"] = {(i+1, j+1):dados["d"][i][j] for j in range(dados["n"]) for i in range(dados["m"])}
ampl.set_option("solver", "highs")
ampl.set_option("highs_options", "timelim=100 debug=1 miploglev=2 outlev=1")
"""ampl.solve(problem='facility',
           solver='highs',
           verbose=True,
           return_output=True,
           timelim=120,
           miploglev=2,
           outlev=2,
           bestbound=1,
           debug=1,
           )
"""
ampl.solve()
custo = ampl.get_objective("custo").value()
y = ampl.get_variable("y").get_values().to_list()
x = ampl.get_variable("x").get_values().to_list()

print(f"Custo: {custo}")
#print(f"Escolas escolhidas: {y}")
#print(f"Designações: {x}")



HiGHS 1.7.0:   lim:time = 100
  tech:debug = 1
  tech:miploglev = 2
  tech:outlev = 1
Running HiGHS 1.7.0 (git hash: 50670fd): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [1e+00, 5e+01]
  Cost   [1e+00, 1e+00]
  Bound  [1e+00, 1e+00]
  RHS    [1e+00, 4e+02]
Presolving model
505051 rows, 250051 cols, 1500050 nonzeros  1s
505051 rows, 250051 cols, 1500050 nonzeros  17s
Objective function is integral with scale 1

Solving MIP model with:
   505051 rows
   250051 cols (250050 binary, 0 integer, 1 implied int., 0 continuous)
   1500050 nonzeros

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
     Proc. InQueue |  Leaves   Expl. | BestBound       BestSol              Gap |   Cuts   InLp Confl. | LpIters     Time

         0       0         0   0.00%   0               inf                  inf        0      0      0         0    21.0s
 R       0       0         0   0.00%   0            

## Passo 5: Quais as possibilidades e limitações?


## Passo 6: Extra

Vamos agora avaliar se nossa solução é eficiente para tamanhos de instâncias diferentes para este problema, levando nossos arquivos para um servidor e deixando a máquina trabalhar sem necessidade de nossa intervenção.

Para logar no terminal remotamente:

```shell
ssh -p <porta> <usuario>@<ip>
```

Podemos usar o VS Code para isso também. Podemos editar o código diretamento no servidor e navegar remotamento nos arquivosl lá contidos.

Nosso servidor para experimentos:

* Usuário: *grupo<n.º>*
* Porta: *9922*
* IP Interno (dentro da UF): *10.3.224.82*
* IP Externo (fora da UF): *177.20.146.65*

Senha e usuário deverão ser obtidos junto ao professor para cada grupo.

## Relatório

O relatório deve conter:

### Título do Problema ou Projeto de Pesquisa

Título que dê ideia do que foi feito.

### Introdução

Explicar em linhas gerais qual o problema que se quer resolver, se é um problema do mundo real, considerando uma empresa ou situação em particular, ou um problema clássico da literatura. Descreve-se de forma textual os objetivos e dados disponíveis, junto com uma justificativa do porquê é importante explorar este problema ou mesmo propor o modelo em questão.

### Formulação do Problema

Apresentar parâmetros, variáveis de decisão, função objetivo e restrições. Apresentar uma explicação breve do que cada expressão formulada representa.

### Experimentos Computacionais

#### Instâncias

Descrever as características dos dados de entrada que serão usados no experimento. Se são dados reais, se são dados simulados, se são dados proveninetes do trabalho de algum autor. Descrever o tamanho das instâncias. No caso do exemplo de hoje, poderíamos descrever em formato de tabela quantos candidatos, quantos locais e o limite de locais a serem usados como características das instâncias. Uma tabela com estas informações é bem-vinda junto ao texto.

#### Recursos computacionais

Descrever o computador utilizado no experimento, em termos de poder de processamento e memória. Descrever também como o modelo foi codificado, em quais linguagens.

#### Configuração do experimento

Descrever o *solver* utilizado e como ele foi configurado, como por exemplo o número de *threads* paralelas permitidas e o tempo limite de processamento. Descrever, se for o caso, quantas vezes uma mesma instância foi executada, caso tenha usado tempos de execução e número de *threads* diferentes em rodada de experimentos diferentes. Podemos testar uma mesma instâncias diversas vezes com configurações diferentes. Analisar a eficiência de uma solução de problema de otimização é trabalhoso, leva tempo e muitas horas de processamento até que se chegue em algo confiável.

#### Resultados

Tabela contendo a identificação das instâncias, as configurações com as quais elas foram resolvidas e os seguintes resultados:

* Custo da função objetivo (Limitante Superior - UB);
* Limitante Inferior (LB), ou custo da relaxação do problema, dada pelo custo dual da função objetivo;
* O GAP, que é a distância percentual entre os dois valores anterires;
* Tempo total de execução;
* Status: Se a solução obtida foi a solução ótima ou não, isto é, o GAP é zero;

Veja um exemplo abaixo:

|Instância|Tempo Limite|Threads|Solver|Custo|LB|GAP|Tempo|Status|
|---|---|---|---|---|---|---|---|---|
|Instância 1|120|1|Highs|376.2|376.2|0.0%|65|Ótima|

#### Análise dos resultados

Discutir se os resultados foram satisfatórios em termos de obtenção da solução, otimalidade e tempo gasto, bem como processamento gasto. Analisar, caso não tenham sido o porquê. Analisar se a solução gerada, mesmo não sendo ótima, poderia ser usada em um cenário real. Discutir quais seriam os próximos passos a serem seguidos no trabalho com este problema.